# 20: LLMs for SWE Productivity
### System Design, Claude Code, GitHub Copilot — and Building Our Own

---

Large Language Models aren't just chat interfaces anymore — they sit **inside the coding loop**.  
Every time you accept a Copilot ghost-text suggestion or ask Claude Code to refactor a function from your terminal, an LLM just ran inference on your codebase.

In this notebook we'll:

1. Understand the **system design** behind tools like Claude Code and GitHub Copilot
2. Experience the **pain** of messy code — then let AI fix it
3. Build up from a single refactor to **semantic search**, **style enforcement**, and **evaluation**
4. Ship a **Flask mini-app** called **DevAssist** that ties everything together

---

### The Story

> You just joined a startup as lead engineer. The previous developer left behind a codebase of utility functions — **no docstrings, cryptic variable names, zero tests**. Your job is to clean it up, document it, and make it maintainable. Doing this manually across hundreds of files is brutal. Let's build AI-powered tools to help.

---

## Setup

We load our API key from a `.env` file and set up the OpenAI client.  
We'll use the **Responses API** (`client.responses.create()`) throughout.

In [17]:
# !pip install openai python-dotenv pandas
import pandas as pd
import os, json, time
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import textwrap
import requests

import requests, json, textwrap

# Create a session that ignores proxy env vars (bypasses VPN for local Ollama)
SESSION = requests.Session()
SESSION.trust_env = False  # equivalent of ollama.Client(..., trust_env=False)



import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

#MODEL = 'gpt-5-nano'
#MODEL = 'gpt-4o-mini'
#MODEL = 'gpt-5-mini'
MODEL = 'gpt-4o'
EMBED_MODEL = "text-embedding-3-small"



client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

print(f"Using model: {MODEL}, Ready to go!")

API key loaded successfully.
OpenAI client ready.
Using model: gpt-4o, Ready to go!


---

## Part 1 — The Mess We Inherited

Here's a taste of the codebase the previous developer left behind.  
Spend 30 seconds reading this. Would you want to maintain it?

In [18]:
# ── The messy legacy codebase ──────────────────────────────────

LEGACY_CODEBASE = {
    "math_helpers.py": '''
def calc(a,b,op):
    if op=="add": return a+b
    elif op=="sub": return a-b
    elif op=="mul": return a*b
    elif op=="div":
        if b==0: return "err"
        return a/b
    return None

def avg(lst):
    s=0
    for i in lst: s+=i
    return s/len(lst)

def fib(n):
    if n<=1: return n
    return fib(n-1)+fib(n-2)
''',

    "string_helpers.py": '''
def fmt(s,w):
    r=""
    for i in range(0,len(s),w):
        r+=s[i:i+w]+"\\n"
    return r

def cnt(s,c):
    t=0
    for x in s:
        if x==c: t+=1
    return t

def rev(s): return s[::-1]

def is_pal(s): return s==s[::-1]
''',

    "data_helpers.py": '''
def flatten(lst):
    r=[]
    for i in lst:
        if type(i)==list: r.extend(flatten(i))
        else: r.append(i)
    return r

def dedup(lst):
    s=set()
    r=[]
    for i in lst:
        if i not in s:
            s.add(i)
            r.append(i)
    return r

def chunk(lst,n):
    return [lst[i:i+n] for i in range(0,len(lst),n)]

def merge_dicts(a,b):
    r=dict(a)
    for k,v in b.items(): r[k]=v
    return r
''',

    "api_helpers.py": '''
import json, time

def retry(fn, tries=3):
    for i in range(tries):
        try: return fn()
        except: time.sleep(1)
    return None

def parse_resp(r):
    try: return json.loads(r)
    except: return {"error": "bad json"}

def build_url(base, params):
    p = "&".join(f"{k}={v}" for k,v in params.items())
    return f"{base}?{p}"
'''
}

# Let's look at one file
print("── math_helpers.py ──")
print(LEGACY_CODEBASE["math_helpers.py"])

── math_helpers.py ──

def calc(a,b,op):
    if op=="add": return a+b
    elif op=="sub": return a-b
    elif op=="mul": return a*b
    elif op=="div":
        if b==0: return "err"
        return a/b
    return None

def avg(lst):
    s=0
    for i in lst: s+=i
    return s/len(lst)

def fib(n):
    if n<=1: return n
    return fib(n-1)+fib(n-2)



No docstrings. Single-letter variable names. No type hints. No error handling strategy.  
**Imagine 500 files like this.** That's the reality Claude Code and Copilot were built to help with.

Before we start building solutions, let's understand how the tools behind the curtain actually work.

---

## Part 2 — System Design: What Makes an AI Coding Tool Tick?

When you type in Claude Code or accept a Copilot suggestion, here's what happens behind the scenes:

```
Keystroke / Command → Gather Context → Send to LLM → Stream Response → Show Output
         │                  │                │               │               │
         │            Which files?      Latency budget   Token streaming   User accepts?
         │            CLAUDE.md?        < 200ms ideal    First token fast  Feedback loop
         │            Open tabs?        Cost per call                        │
         │                  │                                                │
         └──────────────────┴────────── Privacy: does code leave machine? ───┘
```

### The Five Design Pillars

| Pillar | Why It Matters | Real-World Trade-off |
|--------|---------------|---------------------|
| **Latency** | Engineers expect near-instant suggestions | Smaller/faster model vs. smarter but slower |
| **Context** | Model must "see" enough code to be useful | Bigger context window = more tokens = more cost |
| **Cost** | Continuous completions burn API credits fast | Cache embeddings, batch requests, use cheaper models for simple tasks |
| **Privacy** | Source code is proprietary IP | On-device inference vs. cloud API calls |
| **Feedback Loop** | Accepted/rejected suggestions improve the system | Telemetry vs. user consent |

These five pillars are in constant tension. Claude Code and Copilot make different trade-offs — let's compare.

### Claude Code — Key Features

Claude Code is Anthropic's **command-line agentic coding tool** — it lives in your terminal and operates directly on your file system.

- **Agentic workflow** — reads files, writes code, runs commands, and iterates autonomously
- **Full codebase awareness** — can navigate your entire repo, not just the current file
- **`CLAUDE.md`** — a project-level config file (like a README for the AI) that stores conventions, commands, architecture notes, and coding standards. Claude reads this automatically at the start of every session.
- **Multi-file edits** — can refactor across many files in one pass
- **MCP (Model Context Protocol)** — connects to external tools (GitHub, Slack, databases) via standardized protocol
- **Slash commands** — custom reusable workflows (`/test`, `/review`, `/deploy`)
- **Hooks** — pre/post tool-use scripts (e.g., auto-lint after every edit)

### GitHub Copilot — Key Features

- **Autocomplete-style suggestions** — like Gmail's "smart compose" but for code
- **Whole function generation** — write a comment, get the implementation
- **Context from open files** — reads your current file + nearby tabs
- **IDE integration** — VSCode, JetBrains, Neovim
- **Powered by Codex / GPT-4 class models**

### Key Difference

| | Claude Code | Copilot |
|---|---|---|
| **Interface** | Terminal (agentic) | IDE inline (autocomplete) |
| **Scope** | Full repo + terminal commands | Current file + open tabs |
| **Style config** | `CLAUDE.md` | Limited (editor settings) |
| **Workflow** | Plans → executes → validates | Suggests as you type |
| **External tools** | MCP servers | GitHub ecosystem |

### Honest Pros & Cons

| Pros | Cons |
|------|------|
| Massive speed boost on boilerplate | Can suggest incorrect or insecure code |
| Helps new engineers onboard faster | Over-reliance → engineers stop thinking critically |
| Discovers hidden APIs / library methods | Style drift across teams if unconfigured |
| Reduces context-switching | Licensing / ownership ambiguity on generated code |

### 🧠 Quiz 1 — System Design & Tool Landscape

**Q1:** Which factor is *most critical* for developer adoption of AI coding tools?  
a) High accuracy only  
b) Low latency + context awareness  
c) Large model size  

**Q2:** Why is privacy a unique concern in SWE productivity tools?  
a) Source code is proprietary IP  
b) Models cannot process plain text  
c) APIs don't support encryption  

**Q3:** Claude Code's `CLAUDE.md` file is used for:  
a) Version control  
b) Storing project conventions, commands, and coding standards for the AI  
c) API authentication  

**Q4:** What's a key difference between Claude Code and Copilot?  
a) Claude Code only works with Python  
b) Claude Code is agentic (terminal-based, full repo access) while Copilot is inline autocomplete  
c) Copilot can edit multiple files at once  

<details><summary>Click for answers</summary>

1. **b)** — Devs won't use it if it's slow or clueless about their codebase.  
2. **a)** — Leaking proprietary source code is a serious company risk.  
3. **b)** — `CLAUDE.md` is the project-level config that tells Claude Code about your architecture, commands, coding standards — it's loaded at the start of every session.  
4. **b)** — Claude Code operates from the terminal with full repo + command access; Copilot suggests inline within your IDE.

</details>

---

## Part 3 — Our First AI Refactor

Let's start simple. One messy function → one clean function.  
This is the atomic unit of what Claude Code and Copilot do.

In [19]:
# ── The ugly function we want to fix ──────────────────────────

ugly_function = '''
def calc(a,b,op):
    if op=="add": return a+b
    elif op=="sub": return a-b
    elif op=="mul": return a*b
    elif op=="div":
        if b==0: return "err"
        return a/b
    return None
'''

print("BEFORE:")
print(ugly_function)

BEFORE:

def calc(a,b,op):
    if op=="add": return a+b
    elif op=="sub": return a-b
    elif op=="mul": return a*b
    elif op=="div":
        if b==0: return "err"
        return a/b
    return None



In [20]:
MODEL

'gpt-4o'

In [21]:
# ── Ask the LLM to refactor it ───────────────────────────────

response = client.responses.create(
    model=MODEL,
    instructions="You are a senior Python engineer. Return ONLY the refactored code, no explanation.",
    input=f"""Refactor this function to follow Python best practices:
- Add type hints
- Add a clear docstring
- Use descriptive variable names
- Handle errors properly (raise exceptions, don't return strings)

```python
{ugly_function}
```"""
)

print("AFTER:")
print(response.output_text)

AFTER:
```python
def calculate(a: float, b: float, operation: str) -> float:
    """
    Perform a basic arithmetic operation on two numbers.

    Parameters:
    a (float): The first operand.
    b (float): The second operand.
    operation (str): The operation to perform ('add', 'sub', 'mul', 'div').

    Returns:
    float: The result of the operation.

    Raises:
    ValueError: If the operation is not one of the specified strings.
    ZeroDivisionError: If division by zero is attempted.
    """
    if operation == "add":
        return a + b
    elif operation == "sub":
        return a - b
    elif operation == "mul":
        return a * b
    elif operation == "div":
        if b == 0:
            raise ZeroDivisionError("Division by zero is undefined.")
        return a / b
    else:
        raise ValueError("Invalid operation specified.")
```


That's the **happy path** — a single isolated function. But what about functions that **depend on each other**? Let's see the model struggle.

In [22]:
# ── The context problem: refactoring WITHOUT seeing dependencies ─

# Imagine this function calls `calc` from math_helpers.py
dependent_code = '''
def process_order(items):
    total = 0
    for item in items:
        subtotal = calc(item["price"], item["qty"], "mul")
        tax = calc(subtotal, 0.08, "mul")
        total = calc(total, calc(subtotal, tax, "add"), "add")
    return total
'''

# Ask model to refactor WITHOUT showing it calc()
response_no_context = client.responses.create(
    model=MODEL,
    instructions="You are a senior Python engineer. Return ONLY the refactored code.",
    input=f"""Refactor this function to follow Python best practices.
Add type hints, docstring, proper error handling.

```python
{dependent_code}
```"""
)

print("Without context (model doesn't know about calc()):")
print(response_no_context.output_text)

Without context (model doesn't know about calc()):
```python
from typing import List, Dict, Union

def calc(value1: float, value2: float, operation: str) -> float:
    if operation == "mul":
        return value1 * value2
    elif operation == "add":
        return value1 + value2
    else:
        raise ValueError("Unsupported operation")

def process_order(items: List[Dict[str, Union[float, int]]]) -> float:
    """
    Processes an order by calculating the total cost including tax.

    Args:
        items (List[Dict[str, Union[float, int]]]): A list of items, each with 'price' and 'qty'.

    Returns:
        float: The total cost of the order including tax.

    Raises:
        ValueError: If an unsupported operation is requested in calc.
        KeyError: If an item is missing 'price' or 'qty' keys.
    """
    total = 0.0
    for item in items:
        try:
            subtotal = calc(float(item["price"]), int(item["qty"]), "mul")
            tax = calc(subtotal, 0.08, "mul")
  

In [23]:
# ── Now WITH context: show the model both files ───────────────

response_with_context = client.responses.create(
    model=MODEL,
    instructions="You are a senior Python engineer. Return ONLY the refactored code.",
    input=f"""Refactor the `process_order` function to follow Python best practices.
Add type hints, docstring, proper error handling.
Replace calls to the legacy `calc()` with direct Python arithmetic.

Here is the dependency — the `calc` function it currently uses:
```python
{LEGACY_CODEBASE['math_helpers.py']}
```

Here is the function to refactor:
```python
{dependent_code}
```"""
)

print("With context (model sees calc()):")
print(response_with_context.output_text)

With context (model sees calc()):
```python
from typing import List, Dict, Union

def process_order(items: List[Dict[str, Union[int, float]]]) -> float:
    """
    Processes a list of items to calculate the total order cost,
    including tax.

    Args:
        items: A list of dictionaries, each containing 'price' and 'qty'.

    Returns:
        The total cost as a float.
    """
    total = 0.0
    for item in items:
        try:
            price = item["price"]
            qty = item["qty"]
            subtotal = price * qty
            tax = subtotal * 0.08
            total += subtotal + tax
        except KeyError as e:
            raise ValueError(f"Missing key in item: {e}")
        except TypeError:
            raise ValueError(f"Invalid item data: {item}")
    return total
```


**See the difference?**  
Without context, the model has to guess what `calc()` does. With context, it can confidently replace it with clean arithmetic.

This is exactly why **Claude Code navigates your entire repo** and **Copilot reads your open tabs**. Context is everything.

But there's a problem: you can't shove 500 files into a single prompt. That's where **semantic search** comes in.

### 🧠 Quiz 2 — Refactoring Basics

**Q1:** Why did the model produce a better refactor when given the `calc()` source code?  
a) It used a bigger model  
b) It had the dependency context to understand what `calc()` does  
c) It ran unit tests  

**Q2:** What's the risk if we apply AI refactors blindly?  
a) Faster builds  
b) Incorrect or breaking changes  
c) Better autocomplete  

**Q3:** Why can't we just send the entire codebase in every prompt?  
a) Models don't accept text  
b) Token limits, cost, and latency  
c) There's no reason — we should always do this  

<details><summary>Click for answers</summary>

1. **b)** — With dependency context, the model understood what `calc()` does and could replace it intelligently.  
2. **b)** — AI-generated code can introduce subtle bugs; always review and test.  
3. **b)** — Context windows have token limits, sending everything is slow and expensive.

</details>

---

## Part 4 — Semantic Search: Finding the Right Context

Here's the core insight behind how Claude Code and similar tools find relevant code:

```
User types a request
   → Convert to embedding
   → Compare against pre-computed file embeddings
   → Retrieve the most relevant files
   → Inject them into the prompt as context
   → Send to LLM
```

This is basically **RAG (Retrieval-Augmented Generation) for code**. Let's build it.

In [24]:
# ── Step 1: Embed every file in our legacy codebase ──────────

def embed_text(text: str) -> list[float]:
    """Get embedding vector for a piece of text."""
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return response.data[0].embedding


def embed_codebase(codebase: dict[str, str]) -> dict[str, list[float]]:
    """Embed all files in a codebase dictionary."""
    embeddings = {}
    for filename, code in codebase.items():
        embeddings[filename] = embed_text(f"File: {filename}\n{code}")
        print(f"  Embedded {filename} → {len(embeddings[filename])} dimensions")
    return embeddings


print("Embedding the legacy codebase...")
codebase_embeddings = embed_codebase(LEGACY_CODEBASE)
print(f"\nDone! {len(codebase_embeddings)} files indexed.")

Embedding the legacy codebase...
  Embedded math_helpers.py → 1536 dimensions
  Embedded string_helpers.py → 1536 dimensions
  Embedded data_helpers.py → 1536 dimensions
  Embedded api_helpers.py → 1536 dimensions

Done! 4 files indexed.


In [25]:
for key, value in LEGACY_CODEBASE.items():
    print(key, pretty_print(value))
    print("-" * 40)

 def calc(a,b,op):     if op=="add": return a+b     elif op=="sub": return a-b
elif op=="mul": return a*b     elif op=="div":         if b==0: return "err"
return a/b     return None  def avg(lst):     s=0     for i in lst: s+=i
return s/len(lst)  def fib(n):     if n<=1: return n     return
fib(n-1)+fib(n-2)
math_helpers.py None
----------------------------------------
 def fmt(s,w):     r=""     for i in range(0,len(s),w):         r+=s[i:i+w]+"\n"
return r  def cnt(s,c):     t=0     for x in s:         if x==c: t+=1     return
t  def rev(s): return s[::-1]  def is_pal(s): return s==s[::-1]
string_helpers.py None
----------------------------------------
 def flatten(lst):     r=[]     for i in lst:         if type(i)==list:
r.extend(flatten(i))         else: r.append(i)     return r  def dedup(lst):
s=set()     r=[]     for i in lst:         if i not in s:             s.add(i)
r.append(i)     return r  def chunk(lst,n):     return [lst[i:i+n] for i in
range(0,len(lst),n)]  def merge_d

In [26]:
# ── Step 2: Semantic search — find the most relevant file ────

def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def search_codebase(query: str, embeddings: dict, codebase: dict, top_k: int = 2) -> list[tuple]:
    """Search the codebase for files most relevant to a natural language query."""
    query_embedding = embed_text(query)
    
    scores = [
        (filename, cosine_similarity(query_embedding, file_embedding))
        for filename, file_embedding in embeddings.items()
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    
    results = []
    for filename, score in scores[:top_k]:
        results.append((filename, score, codebase[filename]))
    
    return results


# Try several queries
test_queries = [
    "function to greet a user by name",
    "how to retry a failed API call",
    "calculate fibonacci numbers",
    "remove duplicates from a list",
]

for query in test_queries:
    results = search_codebase(query, codebase_embeddings, LEGACY_CODEBASE, top_k=1)
    best_file, score, _ = results[0]
    print(f"Query: {query!r:45s} → {best_file} (score: {score:.3f})")

Query: 'function to greet a user by name'            → string_helpers.py (score: 0.168)
Query: 'how to retry a failed API call'              → api_helpers.py (score: 0.462)
Query: 'calculate fibonacci numbers'                 → math_helpers.py (score: 0.299)
Query: 'remove duplicates from a list'               → data_helpers.py (score: 0.344)


The search correctly routes natural language queries to the right files.  
This is the same principle behind how Claude Code finds relevant context before making changes — it reads and navigates files to understand what's relevant.

In [27]:
# ── Step 3: Context-aware refactoring pipeline ────────────────
# Combine search + refactor into one function

def smart_refactor(task: str, codebase: dict, embeddings: dict, top_k: int = 2) -> str:
    """
    1. Search the codebase for relevant files
    2. Build a context-rich prompt
    3. Ask the LLM to refactor
    """
    # Retrieve relevant context
    results = search_codebase(task, embeddings, codebase, top_k=top_k)
    
    context_block = ""
    for filename, score, code in results:
        context_block += f"\n### {filename} (relevance: {score:.2f})\n```python\n{code}\n```\n"
    
    print(f"Retrieved context from: {[r[0] for r in results]}")
    
    response = client.responses.create(
        model=MODEL,
        instructions="You are a senior Python engineer. Return ONLY the refactored code with docstrings and type hints.",
        input=f"""Task: {task}

Here are the relevant files from the codebase for context:
{context_block}

Refactor the relevant code to follow Python best practices."""
    )
    
    return response.output_text


# Test it
result = smart_refactor(
    "Clean up the data helper functions — add docstrings, type hints, and proper error handling",
    LEGACY_CODEBASE,
    codebase_embeddings
)
print(result)

Retrieved context from: ['data_helpers.py', 'string_helpers.py']
```python
from typing import Any, Dict, List, Union

def flatten(lst: List[Any]) -> List[Any]:
    """
    Flattens a nested list into a single list.

    Args:
        lst: A list which might contain nested lists.

    Returns:
        A flattened list containing all the elements.
    """
    result = []
    for i in lst:
        if isinstance(i, list):
            result.extend(flatten(i))
        else:
            result.append(i)
    return result

def dedup(lst: List[Any]) -> List[Any]:
    """
    Removes duplicate elements from a list while preserving order.

    Args:
        lst: A list to deduplicate.

    Returns:
        A list with duplicates removed.
    """
    seen = set()
    result = []
    for i in lst:
        if i not in seen:
            seen.add(i)
            result.append(i)
    return result

def chunk(lst: List[Any], n: int) -> List[List[Any]]:
    """
    Splits list into chunks of size n.

   

We now have a pipeline that:
1. **Searches** the codebase semantically
2. **Retrieves** relevant files as context
3. **Refactors** with full awareness of dependencies

This is the core loop that powers context-aware coding tools.

### 🧠 Quiz 3 — Semantic Search & Context

**Q1:** Why use embeddings in coding tools?  
a) To reduce latency  
b) To enable semantic search across the codebase  
c) To optimize GPU usage  

**Q2:** In our `smart_refactor` pipeline, what happens first?  
a) The LLM generates code  
b) We search for relevant files using embeddings  
c) We run unit tests  

**Q3:** What does "context window" refer to?  
a) The IDE window  
b) How much text/code the model can "see" in one request  
c) The number of open browser tabs  

<details><summary>Click for answers</summary>

1. **b)** — Embeddings let us find semantically relevant code across hundreds of files.  
2. **b)** — Retrieve first, then generate — this is the RAG pattern applied to code.  
3. **b)** — The context window is the token limit for a single LLM request.

</details>

---

## Part 5 — Style Enforcement (The `CLAUDE.md` Pattern)

Here's a problem: your team has coding standards, but the LLM doesn't know them.  
Without style rules, AI-generated code drifts from your conventions.

**Claude Code** solves this with **`CLAUDE.md`** — a markdown file in your project root that gets loaded automatically at the start of every session. It tells Claude about your:
- Project architecture and directory structure
- Coding standards and style rules
- Build/test/lint commands
- Domain-specific terminology
- Workflow conventions

Think of it as **onboarding documentation for the AI** — the same way you'd onboard a new engineer, except it's read every single session.

Let's build the same pattern.

In [28]:
# ── First: refactor WITHOUT style rules ──────────────────────

response_no_style = client.responses.create(
    model=MODEL,
    instructions="You are a Python engineer. Return ONLY the refactored code.",
    input=f"""Refactor this function:

```python
def avg(lst):
    s=0
    for i in lst: s+=i
    return s/len(lst)
```"""
)

print("Without style rules:")
print(response_no_style.output_text)

Without style rules:
```python
def avg(lst):
    return sum(lst) / len(lst)
```


In [29]:
# ── Define our team's CLAUDE.md-style rules ──────────────────

TEAM_STYLE_RULES = """
# CLAUDE.md — Team Coding Standards

## Code Style
1. ALL functions must use Google-style docstrings with Args, Returns, Raises sections
2. ALL functions must have type hints on parameters AND return type
3. Use descriptive variable names — no single letters (except loop vars like `i`, `j`)
4. Raise specific exceptions (ValueError, TypeError) — NEVER return error strings
5. Add input validation at the start of every public function
6. Use f-strings for string formatting
7. Maximum function length: 20 lines (excluding docstring)
8. Prefer list comprehensions over manual loops when clear
9. All numeric functions must handle empty input gracefully
10. Use snake_case for functions and variables

## Commands
- `pytest tests/` — run all tests
- `ruff check .` — lint

## Architecture
- `/helpers` — pure utility functions (no side effects)
- `/api` — external API wrappers
"""

print(TEAM_STYLE_RULES)


# CLAUDE.md — Team Coding Standards

## Code Style
1. ALL functions must use Google-style docstrings with Args, Returns, Raises sections
2. ALL functions must have type hints on parameters AND return type
3. Use descriptive variable names — no single letters (except loop vars like `i`, `j`)
4. Raise specific exceptions (ValueError, TypeError) — NEVER return error strings
5. Add input validation at the start of every public function
6. Use f-strings for string formatting
7. Maximum function length: 20 lines (excluding docstring)
8. Prefer list comprehensions over manual loops when clear
9. All numeric functions must handle empty input gracefully
10. Use snake_case for functions and variables

## Commands
- `pytest tests/` — run all tests
- `ruff check .` — lint

## Architecture
- `/helpers` — pure utility functions (no side effects)
- `/api` — external API wrappers



In [30]:
# ── Now refactor WITH style rules (simulating CLAUDE.md) ─────

response_with_style = client.responses.create(
    model=MODEL,
    instructions=f"""You are a senior Python engineer.
You MUST follow these team coding standards exactly:

{TEAM_STYLE_RULES}

Return ONLY the refactored code.""",
    input=f"""Refactor this function:

```python
def avg(lst):
    s=0
    for i in lst: s+=i
    return s/len(lst)
```"""
)

print("With CLAUDE.md style rules:")
print(response_with_style.output_text)

With CLAUDE.md style rules:
```python
def calculate_average(numbers: list[float]) -> float:
    """Calculate the average of a list of numbers.

    Args:
        numbers: A list of numeric values.

    Returns:
        The average of the numbers.

    Raises:
        ValueError: If the input list is empty.
    """
    if not numbers:
        raise ValueError("The list of numbers must not be empty.")

    total_sum = sum(numbers)
    return total_sum / len(numbers)
```


Notice the difference: with style rules, the output follows Google-style docstrings, has input validation, raises proper exceptions, and uses descriptive names.

In production, **Claude Code reads `CLAUDE.md` automatically and loads it into context at the start of every session** — same principle, zero friction. You check it into git so your whole team benefits.

In [31]:
# ── Upgrade our smart_refactor to include style rules ────────

def smart_refactor_v2(
    task: str,
    codebase: dict,
    embeddings: dict,
    style_rules: str = "",
    top_k: int = 2
) -> str:
    """
    Context-aware, style-enforced refactoring.
    Combines: semantic search + CLAUDE.md style rules + LLM generation.
    """
    # Retrieve relevant context
    results = search_codebase(task, embeddings, codebase, top_k=top_k)
    
    context_block = ""
    for filename, score, code in results:
        context_block += f"\n### {filename}\n```python\n{code}\n```\n"
    
    # Build system prompt with style rules (just like CLAUDE.md is loaded)
    system_prompt = "You are a senior Python engineer. Return ONLY the refactored code."
    if style_rules:
        system_prompt += f"\n\nFollow these team coding standards:\n{style_rules}"
    
    response = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=f"""Task: {task}

Relevant codebase files:
{context_block}

Refactor the code as requested."""
    )
    
    return response.output_text


# Test the full pipeline
result = smart_refactor_v2(
    "Refactor the string helper utilities",
    LEGACY_CODEBASE,
    codebase_embeddings,
    style_rules=TEAM_STYLE_RULES
)
print(result)

```python
# string_helpers.py

def format_string(s: str, width: int) -> str:
    """Formats the string `s` with line breaks every `width` characters.

    Args:
        s: The input string to be formatted.
        width: The number of characters per line.

    Returns:
        A formatted string with line breaks at the specified width.

    Raises:
        ValueError: If the width is less than or equal to zero.
    """
    if width <= 0:
        raise ValueError("Width must be greater than zero.")
        
    return "\n".join(s[i:i + width] for i in range(0, len(s), width))

def count_characters(s: str, char: str) -> int:
    """Counts the occurrences of `char` in the string `s`.

    Args:
        s: The string in which to count characters.
        char: The character to count.

    Returns:
        The number of occurrences of `char` in `s`.
    """
    return sum(1 for x in s if x == char)

def reverse_string(s: str) -> str:
    """Reverses the input string `s`.

    Args:
        

### 🧠 Quiz 4 — Style Enforcement

**Q1:** What problem does `CLAUDE.md` solve?  
a) Faster model inference  
b) Consistent coding style and project context across AI sessions  
c) Automatic deployment  

**Q2:** In our implementation, where do style rules get injected?  
a) In the user message  
b) In the system instructions  
c) In the embedding model  

**Q3:** Without style rules, what's a common problem with AI-generated code on a team?  
a) Code runs slower  
b) Style drift — each suggestion follows a different convention  
c) The model refuses to generate code  

<details><summary>Click for answers</summary>

1. **b)** — `CLAUDE.md` gives the AI persistent project context: architecture, conventions, commands. Every session starts informed.  
2. **b)** — Style rules go into system instructions so they shape every response.  
3. **b)** — Without rules, each output may use different docstring styles, naming, etc.

</details>

---

## Part 6 — Documentation Generation

Refactoring is only half the battle. We also need **documentation**.  
Let's build a doc generator that reads code and produces markdown documentation — the kind you'd put in a README or a wiki.

In [32]:
# ── Generate documentation for a file ────────────────────────

def generate_docs(filename: str, code: str) -> str:
    """Generate markdown documentation for a code file."""
    response = client.responses.create(
        model=MODEL,
        instructions="""You are a technical writer. Generate clean Markdown documentation.
Include:
- Module overview (1-2 sentences)
- Function signatures with parameter descriptions
- Usage examples for each function
- Any important notes or caveats

Return ONLY the Markdown documentation.""",
        input=f"""Generate documentation for this Python module:

Filename: {filename}
```python
{code}
```"""
    )
    return response.output_text


# Generate docs for one file
docs = generate_docs("data_helpers.py", LEGACY_CODEBASE["data_helpers.py"])
print(docs)

# data_helpers.py

This module provides utility functions for manipulating lists and dictionaries, including flattening nested lists, removing duplicates, chunking, and merging dictionaries.

## Function Signatures

### `flatten(lst)`

Flattens a nested list into a single list of values.

- **Parameters:**
  - `lst` (list): A possibly nested list of elements.

- **Returns:**
  - `list`: A flat list containing all elements.

### `dedup(lst)`

Removes duplicates from a list while preserving order.

- **Parameters:**
  - `lst` (list): A list from which duplicates should be removed.

- **Returns:**
  - `list`: A list with duplicates removed.

### `chunk(lst, n)`

Divides a list into chunks of size `n`.

- **Parameters:**
  - `lst` (list): The list to be divided.
  - `n` (int): The chunk size.

- **Returns:**
  - `list of lists`: A list of chunked lists.

### `merge_dicts(a, b)`

Merges two dictionaries, with values from the second dictionary overwriting those from the first in case of key 

In [33]:
# ── Generate docs for the entire codebase ────────────────────

print("Generating documentation for all files...\n")
all_docs = {}
for filename, code in LEGACY_CODEBASE.items():
    print(f"  Documenting {filename}...")
    all_docs[filename] = generate_docs(filename, code)

# Show a summary
print("\n" + "=" * 60)
print("Documentation generated for:")
for filename, doc in all_docs.items():
    line_count = doc.count("\n") + 1
    print(f"  {filename} → {line_count} lines of docs")

Generating documentation for all files...

  Documenting math_helpers.py...
  Documenting string_helpers.py...
  Documenting data_helpers.py...
  Documenting api_helpers.py...

Documentation generated for:
  math_helpers.py → 56 lines of docs
  string_helpers.py → 96 lines of docs
  data_helpers.py → 76 lines of docs
  api_helpers.py → 75 lines of docs


---

## Part 7 — Evaluation: Is the AI Actually Helping?

In production, Claude Code and Copilot track metrics to know if they're useful:

| Metric | What It Measures |
|--------|------------------|
| **Acceptance rate** | % of suggestions the developer keeps |
| **Correctness** | Does the generated code pass tests? |
| **Latency** | Time from request to suggestion |
| **Cost per suggestion** | API token spend per completion |

Let's build a simple evaluation harness.

In [34]:
# ── Evaluation harness: test if refactored code actually works ─

# Define test cases for our legacy functions
TEST_CASES = [
    {
        "name": "calc addition",
        "original_code": "calc(3, 4, 'add')",
        "expected": 7
    },
    {
        "name": "calc division by zero",
        "original_code": "calc(10, 0, 'div')",
        "expected": "err"   # original returns string "err"
    },
    {
        "name": "average of list",
        "original_code": "avg([10, 20, 30])",
        "expected": 20.0
    },
    {
        "name": "fibonacci(6)",
        "original_code": "fib(6)",
        "expected": 8
    },
]

# Run tests against original code
exec(LEGACY_CODEBASE["math_helpers.py"])  # Load functions into namespace

print("Testing original legacy code:\n")
for test in TEST_CASES:
    try:
        result = eval(test["original_code"])
        passed = result == test["expected"]
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {test['name']}: expected={test['expected']}, got={result}")
    except Exception as e:
        print(f"  [ERROR] {test['name']}: {e}")

Testing original legacy code:

  [PASS] calc addition: expected=7, got=7
  [PASS] calc division by zero: expected=err, got=err
  [PASS] average of list: expected=20.0, got=20.0
  [PASS] fibonacci(6): expected=8, got=8


In [35]:
# ── Measure latency and cost of our refactoring calls ────────

def measure_refactor(task: str, code: str) -> dict:
    """Measure the latency and token usage of a refactor call."""
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions="You are a senior Python engineer. Return ONLY the refactored code.",
        input=f"{task}\n\n```python\n{code}\n```"
    )
    
    elapsed = time.time() - start
    
    return {
        "task": task,
        "latency_seconds": round(elapsed, 2),
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
        "output_preview": response.output_text[:100] + "..."
    }


# Run measurements
tasks = [
    ("Add docstrings and type hints", LEGACY_CODEBASE["math_helpers.py"]),
    ("Refactor for readability", LEGACY_CODEBASE["string_helpers.py"]),
    ("Add error handling and validation", LEGACY_CODEBASE["api_helpers.py"]),
]

print("Measuring refactor performance:\n")
for task_desc, code in tasks:
    metrics = measure_refactor(task_desc, code)
    print(f"  Task: {metrics['task']}")
    print(f"    Latency: {metrics['latency_seconds']}s")
    print(f"    Tokens:  {metrics['input_tokens']} in / {metrics['output_tokens']} out / {metrics['total_tokens']} total")
    print()

Measuring refactor performance:

  Task: Add docstrings and type hints
    Latency: 4.16s
    Tokens:  150 in / 329 out / 479 total

  Task: Refactor for readability
    Latency: 2.37s
    Tokens:  121 in / 106 out / 227 total

  Task: Add error handling and validation
    Latency: 3.63s
    Tokens:  135 in / 161 out / 296 total



### Key Evaluation Insights

- **Latency**: Our calls are in the 1-3s range — fine for batch refactoring, but Copilot needs sub-200ms for inline suggestions (they use smaller/faster models and speculative decoding)
- **Token cost**: Even simple refactors consume hundreds of tokens — at scale, this adds up
- **Correctness**: We can't just assume the refactored code works — always run tests

In production systems, these metrics feed into a **feedback loop**: if a model's acceptance rate drops, the team investigates and tunes prompts, context retrieval, or model selection.

### 🧠 Quiz 5 — Evaluation & Risks

**Q1:** How do Copilot / Claude Code measure if AI suggestions are useful?  
a) Count lines of code generated  
b) Track acceptance rates and time saved  
c) Number of API calls  

**Q2:** Which is a major risk of AI coding tools?  
a) Longer code comments  
b) Security vulnerabilities in generated code  
c) More unit tests being written  

**Q3:** Best practice for AI-generated code is:  
a) Deploy directly  
b) Always review and test before merging  
c) Never use in production  

**Q4:** What makes `CLAUDE.md` different from just a README?  
a) It's encrypted  
b) It's loaded into the AI's context automatically every session — it's instructions *for the AI*, not just for humans  
c) It only works with Python  

<details><summary>Click for answers</summary>

1. **b)** — Acceptance rate (did the dev keep the suggestion?) and time saved are the key metrics.  
2. **b)** — AI can generate code with vulnerabilities (SQL injection, missing auth checks, etc.).  
3. **b)** — AI output is a draft, not a final product. Always review and test.  
4. **b)** — `CLAUDE.md` is specifically designed to be machine-read: it onboards the AI into your project every session, automatically.

</details>

---

## Part 8 — Caching: Don't Re-embed What Hasn't Changed

Every time we embed a file, we spend tokens and time. If a file hasn't changed, why re-embed it?  
Real tools cache embeddings on disk and only re-embed when a file is modified.

Let's add a simple hash-based caching layer.

In [36]:
import hashlib

# ── Simple embedding cache ───────────────────────────────────

class EmbeddingCache:
    """Cache embeddings by content hash — skip re-embedding unchanged files."""
    
    def __init__(self):
        self.cache = {}   # hash -> embedding vector
        self.hits = 0
        self.misses = 0
    
    def _hash(self, text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()[:16]
    
    def get_embedding(self, text: str) -> list[float]:
        key = self._hash(text)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        
        self.misses += 1
        embedding = embed_text(text)
        self.cache[key] = embedding
        return embedding
    
    def stats(self) -> str:
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        return f"Cache: {self.hits} hits, {self.misses} misses ({hit_rate:.0f}% hit rate)"


# Demo: embed codebase twice — second time should be all cache hits
cache = EmbeddingCache()

print("First pass (cold cache):")
for filename, code in LEGACY_CODEBASE.items():
    cache.get_embedding(f"File: {filename}\n{code}")
print(f"  {cache.stats()}")

print("\nSecond pass (warm cache — no API calls!):")
for filename, code in LEGACY_CODEBASE.items():
    cache.get_embedding(f"File: {filename}\n{code}")
print(f"  {cache.stats()}")

print("\nSimulated file change (modify one file):")
modified = LEGACY_CODEBASE["math_helpers.py"] + "\n# Updated!\n"
cache.get_embedding(f"File: math_helpers.py\n{modified}")
print(f"  {cache.stats()}")

First pass (cold cache):
  Cache: 0 hits, 4 misses (0% hit rate)

Second pass (warm cache — no API calls!):
  Cache: 4 hits, 4 misses (50% hit rate)

Simulated file change (modify one file):
  Cache: 4 hits, 5 misses (44% hit rate)


On the second pass, **zero API calls** — all embeddings came from cache.  
When a file changes, only that file's embedding is recomputed. In a 10,000-file codebase, this saves massive cost.

In production, this cache would be stored on disk (SQLite, Redis, etc.) and persist across sessions.

---

## Part 9 — Putting It All Together: The DevAssist Flask App

Time to combine everything into a real application. **DevAssist** is a web-based code assistant with:

| Feature | What We Built |
|---------|---------------|
| **Semantic Search** | Find relevant files from natural language queries |
| **Smart Refactor** | Context-aware code cleanup with style enforcement |
| **Doc Generator** | Auto-generate Markdown documentation |
| **Metrics Dashboard** | Track latency and token cost per operation |

Let's write the Flask app.

In [39]:
%%writefile devassist_app.py
"""
DevAssist — AI-Powered Code Assistant
A Flask app combining semantic search, refactoring, doc generation, and metrics.
Demonstrates the same patterns behind Claude Code and GitHub Copilot.
"""

from flask import Flask, request, jsonify, render_template_string
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np
import hashlib
import time

load_dotenv()
app = Flask(__name__)
client = OpenAI()

MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

# ── In-memory codebase & state ────────────────────────────────

CODEBASE = {
    "math_helpers.py": '''def calc(a,b,op):
    if op=="add": return a+b
    elif op=="sub": return a-b
    elif op=="mul": return a*b
    elif op=="div":
        if b==0: return "err"
        return a/b
    return None

def avg(lst):
    s=0
    for i in lst: s+=i
    return s/len(lst)

def fib(n):
    if n<=1: return n
    return fib(n-1)+fib(n-2)''',

    "string_helpers.py": '''def fmt(s,w):
    r=""
    for i in range(0,len(s),w):
        r+=s[i:i+w]+"\\n"
    return r

def cnt(s,c):
    t=0
    for x in s:
        if x==c: t+=1
    return t

def rev(s): return s[::-1]

def is_pal(s): return s==s[::-1]''',

    "data_helpers.py": '''def flatten(lst):
    r=[]
    for i in lst:
        if type(i)==list: r.extend(flatten(i))
        else: r.append(i)
    return r

def dedup(lst):
    s=set()
    r=[]
    for i in lst:
        if i not in s:
            s.add(i)
            r.append(i)
    return r

def chunk(lst,n):
    return [lst[i:i+n] for i in range(0,len(lst),n)]''',

    "api_helpers.py": '''import json, time

def retry(fn, tries=3):
    for i in range(tries):
        try: return fn()
        except: time.sleep(1)
    return None

def parse_resp(r):
    try: return json.loads(r)
    except: return {"error": "bad json"}

def build_url(base, params):
    p = "&".join(f"{k}={v}" for k,v in params.items())
    return f"{base}?{p}"'''
}

# CLAUDE.md-style rules loaded into every AI call
STYLE_RULES = """
# CLAUDE.md — Team Coding Standards

## Code Style
1. Use Google-style docstrings with Args, Returns, Raises sections
2. Add type hints on parameters and return type
3. Use descriptive variable names — no single letters
4. Raise specific exceptions — never return error strings
5. Add input validation at the start of every function
6. Prefer list comprehensions when clear
7. Use snake_case for functions and variables

## Architecture
- /helpers — pure utility functions (no side effects)
- /api — external API wrappers
"""

# Embedding cache
embedding_cache = {}
metrics_log = []


# ── Helper functions ──────────────────────────────────────────

def get_embedding(text):
    key = hashlib.sha256(text.encode()).hexdigest()[:16]
    if key not in embedding_cache:
        resp = client.embeddings.create(model=EMBED_MODEL, input=text)
        embedding_cache[key] = resp.data[0].embedding
    return embedding_cache[key]


def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def search_codebase(query, top_k=2):
    q_emb = get_embedding(query)
    scores = []
    for fname, code in CODEBASE.items():
        f_emb = get_embedding(f"File: {fname}\n{code}")
        scores.append((fname, cosine_sim(q_emb, f_emb), code))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


# ── API Routes ────────────────────────────────────────────────

@app.route("/")
def index():
    return render_template_string(HTML_TEMPLATE)


@app.route("/api/search", methods=["POST"])
def api_search():
    query = request.json.get("query", "")
    start = time.time()
    results = search_codebase(query, top_k=3)
    elapsed = time.time() - start
    
    metrics_log.append({"action": "search", "latency": elapsed, "query": query})
    
    return jsonify({
        "results": [{"file": f, "score": round(s, 3), "code": c} for f, s, c in results],
        "latency_ms": round(elapsed * 1000)
    })


@app.route("/api/refactor", methods=["POST"])
def api_refactor():
    data = request.json
    task = data.get("task", "Refactor this code")
    use_style = data.get("use_style", True)
    use_context = data.get("use_context", True)
    
    start = time.time()
    
    # Build context
    context_block = ""
    retrieved_files = []
    if use_context:
        results = search_codebase(task, top_k=2)
        for fname, score, code in results:
            context_block += f"\n### {fname}\n```python\n{code}\n```\n"
            retrieved_files.append(fname)
    
    # Build system prompt (CLAUDE.md pattern)
    sys_prompt = "You are a senior Python engineer. Return ONLY the refactored code."
    if use_style:
        sys_prompt += f"\n\nTeam coding standards (from CLAUDE.md):\n{STYLE_RULES}"
    
    prompt = f"Task: {task}"
    if context_block:
        prompt += f"\n\nRelevant codebase files:{context_block}"
    prompt += "\n\nRefactor the relevant code."
    
    response = client.responses.create(
        model=MODEL,
        instructions=sys_prompt,
        input=prompt
    )
    
    elapsed = time.time() - start
    
    metrics_log.append({
        "action": "refactor",
        "latency": elapsed,
        "tokens": response.usage.total_tokens,
        "task": task
    })
    
    return jsonify({
        "refactored_code": response.output_text,
        "context_files": retrieved_files,
        "latency_ms": round(elapsed * 1000),
        "tokens_used": response.usage.total_tokens
    })


@app.route("/api/document", methods=["POST"])
def api_document():
    filename = request.json.get("filename", "")
    
    if filename not in CODEBASE:
        return jsonify({"error": f"File '{filename}' not found"}), 404
    
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions="""Generate clean Markdown documentation including:
- Module overview
- Function signatures with parameter descriptions
- Usage examples
Return ONLY the Markdown.""",
        input=f"""Generate documentation for:\n\nFilename: {filename}\n```python\n{CODEBASE[filename]}\n```"""
    )
    
    elapsed = time.time() - start
    
    metrics_log.append({
        "action": "document",
        "latency": elapsed,
        "tokens": response.usage.total_tokens,
        "file": filename
    })
    
    return jsonify({
        "documentation": response.output_text,
        "latency_ms": round(elapsed * 1000),
        "tokens_used": response.usage.total_tokens
    })


@app.route("/api/metrics")
def api_metrics():
    if not metrics_log:
        return jsonify({"message": "No operations recorded yet"})
    
    total_ops = len(metrics_log)
    avg_latency = sum(m["latency"] for m in metrics_log) / total_ops
    total_tokens = sum(m.get("tokens", 0) for m in metrics_log)
    by_action = {}
    for m in metrics_log:
        action = m["action"]
        by_action.setdefault(action, []).append(m["latency"])
    
    return jsonify({
        "total_operations": total_ops,
        "average_latency_ms": round(avg_latency * 1000),
        "total_tokens": total_tokens,
        "cache_size": len(embedding_cache),
        "by_action": {
            action: {
                "count": len(latencies),
                "avg_latency_ms": round(sum(latencies) / len(latencies) * 1000)
            }
            for action, latencies in by_action.items()
        }
    })


@app.route("/api/files")
def api_files():
    return jsonify({"files": list(CODEBASE.keys())})


# ── HTML Template ─────────────────────────────────────────────

HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DevAssist — AI Code Assistant</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            font-family: "SF Mono", "Fira Code", "Consolas", monospace;
            background: #0d1117; color: #c9d1d9;
            padding: 20px; max-width: 1200px; margin: 0 auto;
        }
        h1 { color: #58a6ff; margin-bottom: 5px; font-size: 1.6em; }
        .subtitle { color: #8b949e; margin-bottom: 25px; font-size: 0.9em; }
        .tabs {
            display: flex; gap: 0; margin-bottom: 20px;
            border-bottom: 1px solid #30363d;
        }
        .tab {
            padding: 10px 20px; cursor: pointer; color: #8b949e;
            border-bottom: 2px solid transparent;
            transition: all 0.2s;
        }
        .tab:hover { color: #c9d1d9; }
        .tab.active { color: #58a6ff; border-bottom-color: #58a6ff; }
        .panel { display: none; }
        .panel.active { display: block; }
        .input-group { margin-bottom: 15px; }
        label { display: block; color: #8b949e; margin-bottom: 5px; font-size: 0.85em; }
        input[type="text"], textarea, select {
            width: 100%; padding: 10px; background: #161b22;
            border: 1px solid #30363d; color: #c9d1d9; border-radius: 6px;
            font-family: inherit; font-size: 0.9em;
        }
        textarea { min-height: 80px; resize: vertical; }
        button {
            padding: 10px 20px; background: #238636; color: white;
            border: none; border-radius: 6px; cursor: pointer;
            font-family: inherit; font-weight: 600; font-size: 0.9em;
            transition: background 0.2s;
        }
        button:hover { background: #2ea043; }
        button:disabled { background: #21262d; cursor: wait; }
        .toggle-row { display: flex; gap: 20px; margin-bottom: 15px; }
        .toggle-row label { display: flex; align-items: center; gap: 6px; cursor: pointer; color: #c9d1d9; }
        .result-box {
            background: #161b22; border: 1px solid #30363d;
            border-radius: 6px; padding: 15px; margin-top: 15px;
            white-space: pre-wrap; font-size: 0.85em;
            max-height: 500px; overflow-y: auto;
        }
        .meta { color: #8b949e; font-size: 0.8em; margin-top: 8px; }
        .metrics-grid {
            display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px; margin-top: 15px;
        }
        .metric-card {
            background: #161b22; border: 1px solid #30363d;
            border-radius: 6px; padding: 15px; text-align: center;
        }
        .metric-value { font-size: 1.8em; color: #58a6ff; font-weight: 700; }
        .metric-label { color: #8b949e; font-size: 0.8em; margin-top: 4px; }
        .search-result {
            background: #161b22; border: 1px solid #30363d;
            border-radius: 6px; padding: 12px; margin-top: 10px;
        }
        .search-result .fname { color: #58a6ff; font-weight: 600; }
        .search-result .score { color: #3fb950; float: right; }
        .search-result pre { margin-top: 8px; font-size: 0.82em; color: #8b949e; }
    </style>
</head>
<body>
    <h1>DevAssist</h1>
    <div class="subtitle">AI-Powered Code Assistant &mdash; Semantic Search &bull; Smart Refactor &bull; Doc Gen &bull; Metrics</div>

    <div class="tabs">
        <div class="tab active" onclick="switchTab('search')">Search</div>
        <div class="tab" onclick="switchTab('refactor')">Refactor</div>
        <div class="tab" onclick="switchTab('docs')">Generate Docs</div>
        <div class="tab" onclick="switchTab('metrics')">Metrics</div>
    </div>

    <!-- SEARCH -->
    <div id="panel-search" class="panel active">
        <div class="input-group">
            <label>Describe what you\'re looking for in plain English</label>
            <input type="text" id="search-query" placeholder="e.g. function to retry API calls">
        </div>
        <button onclick="doSearch()">Search Codebase</button>
        <div id="search-results"></div>
    </div>

    <!-- REFACTOR -->
    <div id="panel-refactor" class="panel">
        <div class="input-group">
            <label>What do you want to refactor?</label>
            <textarea id="refactor-task" placeholder="e.g. Clean up the math helper functions — add type hints and docstrings"></textarea>
        </div>
        <div class="toggle-row">
            <label><input type="checkbox" id="use-context" checked> Use semantic context</label>
            <label><input type="checkbox" id="use-style" checked> Enforce CLAUDE.md style rules</label>
        </div>
        <button onclick="doRefactor()">Refactor</button>
        <div id="refactor-results"></div>
    </div>

    <!-- DOCS -->
    <div id="panel-docs" class="panel">
        <div class="input-group">
            <label>Select a file to document</label>
            <select id="doc-file"></select>
        </div>
        <button onclick="doDocs()">Generate Docs</button>
        <div id="doc-results"></div>
    </div>

    <!-- METRICS -->
    <div id="panel-metrics" class="panel">
        <button onclick="doMetrics()">Refresh Metrics</button>
        <div id="metrics-results"></div>
    </div>

    <script>
        // Tab switching
        function switchTab(name) {
            document.querySelectorAll(".tab").forEach(t => t.classList.remove("active"));
            document.querySelectorAll(".panel").forEach(p => p.classList.remove("active"));
            event.target.classList.add("active");
            document.getElementById("panel-" + name).classList.add("active");
        }

        // Load file list
        fetch("/api/files").then(r => r.json()).then(data => {
            const sel = document.getElementById("doc-file");
            data.files.forEach(f => {
                const opt = document.createElement("option");
                opt.value = f; opt.textContent = f;
                sel.appendChild(opt);
            });
        });

        async function doSearch() {
            const query = document.getElementById("search-query").value;
            const btn = event.target; btn.disabled = true; btn.textContent = "Searching...";
            const res = await fetch("/api/search", {
                method: "POST", headers: {"Content-Type": "application/json"},
                body: JSON.stringify({query})
            });
            const data = await res.json();
            btn.disabled = false; btn.textContent = "Search Codebase";
            let html = `<div class="meta">Completed in ${data.latency_ms}ms</div>`;
            data.results.forEach(r => {
                html += `<div class="search-result">
                    <span class="fname">${r.file}</span>
                    <span class="score">Score: ${r.score}</span>
                    <pre>${r.code}</pre>
                </div>`;
            });
            document.getElementById("search-results").innerHTML = html;
        }

        async function doRefactor() {
            const task = document.getElementById("refactor-task").value;
            const use_context = document.getElementById("use-context").checked;
            const use_style = document.getElementById("use-style").checked;
            const btn = event.target; btn.disabled = true; btn.textContent = "Refactoring...";
            const res = await fetch("/api/refactor", {
                method: "POST", headers: {"Content-Type": "application/json"},
                body: JSON.stringify({task, use_context, use_style})
            });
            const data = await res.json();
            btn.disabled = false; btn.textContent = "Refactor";
            let html = `<div class="meta">
                Context from: ${data.context_files.join(", ") || "none"} &bull;
                ${data.latency_ms}ms &bull; ${data.tokens_used} tokens
            </div>`;
            html += `<div class="result-box">${data.refactored_code}</div>`;
            document.getElementById("refactor-results").innerHTML = html;
        }

        async function doDocs() {
            const filename = document.getElementById("doc-file").value;
            const btn = event.target; btn.disabled = true; btn.textContent = "Generating...";
            const res = await fetch("/api/document", {
                method: "POST", headers: {"Content-Type": "application/json"},
                body: JSON.stringify({filename})
            });
            const data = await res.json();
            btn.disabled = false; btn.textContent = "Generate Docs";
            let html = `<div class="meta">${data.latency_ms}ms &bull; ${data.tokens_used} tokens</div>`;
            html += `<div class="result-box">${data.documentation}</div>`;
            document.getElementById("doc-results").innerHTML = html;
        }

        async function doMetrics() {
            const res = await fetch("/api/metrics");
            const data = await res.json();
            if (data.message) {
                document.getElementById("metrics-results").innerHTML =
                    `<div class="result-box">${data.message}</div>`;
                return;
            }
            let html = `<div class="metrics-grid">
                <div class="metric-card">
                    <div class="metric-value">${data.total_operations}</div>
                    <div class="metric-label">Total Operations</div>
                </div>
                <div class="metric-card">
                    <div class="metric-value">${data.average_latency_ms}ms</div>
                    <div class="metric-label">Avg Latency</div>
                </div>
                <div class="metric-card">
                    <div class="metric-value">${data.total_tokens}</div>
                    <div class="metric-label">Total Tokens</div>
                </div>
                <div class="metric-card">
                    <div class="metric-value">${data.cache_size}</div>
                    <div class="metric-label">Cached Embeddings</div>
                </div>
            </div>`;
            if (data.by_action) {
                html += `<div class="result-box">Breakdown by action:\\n`;
                for (const [action, stats] of Object.entries(data.by_action)) {
                    html += `  ${action}: ${stats.count} ops, avg ${stats.avg_latency_ms}ms\\n`;
                }
                html += `</div>`;
            }
            document.getElementById("metrics-results").innerHTML = html;
        }
    </script>
</body>
</html>
'''

if __name__ == "__main__":
    print("Starting DevAssist on http://127.0.0.1:5000")
    app.run(debug=True, port=5000)

Overwriting devassist_app.py


In [38]:
# ── To run the app, uncomment and execute: ───────────────────
# !python devassist_app.py

# Or run in the background:
# import subprocess
# proc = subprocess.Popen(["python", "devassist_app.py"])
# print(f"DevAssist running at http://127.0.0.1:5000  (PID: {proc.pid})")

print("DevAssist app written to devassist_app.py")
print("Run with: python devassist_app.py")
print("Then open: http://127.0.0.1:5000")

DevAssist app written to devassist_app.py
Run with: python devassist_app.py
Then open: http://127.0.0.1:5000


### What DevAssist Demonstrates

| Feature | Mirrors | How |
|---------|---------|-----|
| **Semantic Search** | Claude Code's file navigation | Embeddings + cosine similarity |
| **Smart Refactor** | Copilot's code suggestions | RAG: retrieve context → generate |
| **Style Enforcement** | Claude Code's `CLAUDE.md` | Style rules injected as system instructions |
| **Doc Generation** | Copilot's `/doc` command | LLM reads code, writes markdown |
| **Embedding Cache** | Claude Code's codebase awareness | Hash-based skip for unchanged files |
| **Metrics Dashboard** | Internal telemetry | Latency, token cost, operation counts |

---

## Part 10 — Full Circle: Lessons & Best Practices

We've gone from a single refactor call to a full web app. Here's what we've learned:

### System Design Takeaways

```
1. CONTEXT IS KING
   └─ Without seeing dependencies, the model guesses. With context, it reasons.
   └─ Semantic search (embeddings) is how you scale context to large codebases.

2. STYLE RULES PREVENT DRIFT
   └─ CLAUDE.md / system instructions ensure consistent output.
   └─ Without them, every suggestion follows a different convention.
   └─ Check CLAUDE.md into git so the whole team benefits.

3. CACHING SAVES MONEY
   └─ Don't re-embed unchanged files.
   └─ In production: SQLite/Redis for persistent cache.

4. ALWAYS EVALUATE
   └─ Track latency, token cost, acceptance rate.
   └─ Run tests on AI-generated code before trusting it.

5. PRIVACY IS NON-NEGOTIABLE
   └─ Proprietary code sent to cloud APIs = risk.
   └─ Options: on-device models, PII stripping, private deployments.

6. AI OUTPUT IS A DRAFT, NOT A PRODUCT
   └─ Always review. Always test. Never deploy blindly.
```

### What Claude Code & Copilot Do Beyond What We Built

- **Agentic loops** — Claude Code plans, executes, runs tests, and iterates autonomously
- **Streaming tokens** for real-time ghost text (Copilot)
- **Speculative decoding** for sub-100ms latency
- **MCP (Model Context Protocol)** — Claude Code connects to GitHub, Slack, databases via standardized protocol
- **Hooks** — pre/post tool-use scripts (auto-lint, auto-test after every edit)
- **Multi-file edits** across entire projects
- **Telemetry feedback loops** learning from millions of accept/reject signals

### 🧠 Final Quiz — Putting It All Together

**Q1:** In our DevAssist app, what runs BEFORE the LLM generates a refactored output?  
a) Unit tests  
b) Semantic search to find relevant context files  
c) Code compilation  

**Q2:** Why does our embedding cache use a content hash as the key?  
a) For security  
b) So unchanged files return cached embeddings without an API call  
c) To compress the embedding vectors  

**Q3:** If Copilot's acceptance rate drops from 35% to 20%, what should the team investigate?  
a) Buy more GPUs  
b) Prompt quality, context retrieval, or model selection  
c) Remove the feedback loop  

**Q4:** What's the real-world equivalent of our `STYLE_RULES` string?  
a) A `.gitignore` file  
b) Claude Code's `CLAUDE.md` file  
c) A `requirements.txt` file  

<details><summary>Click for answers</summary>

1. **b)** — The RAG pattern: retrieve first, then generate.  
2. **b)** — If the file content hasn't changed, the hash matches and we skip the API call.  
3. **b)** — A drop in acceptance rate signals something is off in the suggestion pipeline.  
4. **b)** — `CLAUDE.md` is the project-level config that stores conventions, architecture, commands — loaded into every AI session.

</details>

---

## References

- Claude Code: https://claude.com/blog/using-claude-md-files
- Claude Code Best Practices: https://code.claude.com/docs/en/best-practices
- GitHub Copilot: https://github.com/features/copilot
- OpenAI Embeddings: https://platform.openai.com/docs/guides/embeddings
- Technical considerations for AI SWE tools: https://www.innovationendeavors.com/insights/building-ai-powered-software-engineering-tools-essential-technical-considerations-for-founders
- AI IDE Comparative Analysis: https://medium.com/@adnanmasood/inside-the-ai-ide-boom-how-cursor-copilot-and-replit-are-redefining-the-craft-of-code-fe0c4e8ac431